# DSAI 413 — A2: Notebook 4 — Model Comparison

**Goal:** Systematically compare MedGemma, ColPali, and CLIP across all evaluation dimensions.

**Comparisons:**
1. Report generation quality: MedGemma (BLEU/ROUGE/BERTScore)
2. Retrieval quality: ColPali vs CLIP (Precision@K)
3. CLIP image-text alignment: generated reports vs ground truth
4. QA performance: MedGemma with ColPali context vs without
5. Side-by-side qualitative examples
6. Final comparison table

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, Markdown

from src.config import (
    MIMIC_CSV_PATH, MIMIC_IMG_COL, MIMIC_TEXT_COL,
    EVAL_SAMPLE_SIZE, QA_PAIRS_PATH
)
from src.preprocessing import load_mimic_subset, load_image
from src.evaluation import (
    compute_report_metrics, compute_qa_metrics,
    compute_precision_at_k, generate_comparison_table
)

print('Imports OK')

## 1. Load test data and existing evaluation results

In [ ]:
with open('../data/split_indices.json') as f:
    splits = json.load(f)

all_images, all_reports, all_paths = load_mimic_subset(
    MIMIC_CSV_PATH, img_col=MIMIC_IMG_COL, text_col=MIMIC_TEXT_COL
)

test_idx = splits['test'][:EVAL_SAMPLE_SIZE]
test_images  = [all_images[i]  for i in test_idx]
test_reports = [all_reports[i] for i in test_idx]
test_paths   = [all_paths[i]   for i in test_idx]

with open(QA_PAIRS_PATH) as f:
    qa_pairs = json.load(f)

print(f'Test set: {len(test_images)} images | QA pairs: {len(qa_pairs)}')

## 2. MedGemma report generation metrics

In [ ]:
from src.mode1_report_gen import ReportGenerationPipeline

pipeline = ReportGenerationPipeline(use_clip=True, medgemma_load_in_4bit=True)
pipeline.load_models()

results_mode1 = pipeline.run_batch(test_images[:20], ground_truth_reports=test_reports[:20])
gen_reports = [r['report'] for r in results_mode1]

medgemma_report_metrics = compute_report_metrics(gen_reports, test_reports[:20])
print('MedGemma Report Metrics:')
for k, v in medgemma_report_metrics.items():
    print(f'  {k}: {v}')

## 3. Retrieval comparison: ColPali vs CLIP — Precision@K

In [ ]:
# Build two indices — one ColPali, one CLIP
from src.retrieval import build_index_from_images, RetrievalIndex
from src.config import FAISS_INDEX_PATH, FAISS_META_PATH

train_idx    = splits['train']
train_images = [all_images[i]  for i in train_idx]
train_reports= [all_reports[i] for i in train_idx]
train_paths  = [all_paths[i]   for i in train_idx]

# ColPali index (already built in notebook 3 — load it)
colpali_index = RetrievalIndex.load()

# CLIP index
clip_index_path = FAISS_INDEX_PATH.replace('.bin', '_clip.bin')
clip_meta_path  = FAISS_META_PATH.replace('.json', '_clip.json')

if not os.path.exists(clip_index_path):
    clip_index = build_index_from_images(
        train_images, train_reports, train_paths,
        backend='clip',
        save=False,  # save manually to clip-specific path
    )
    clip_index.save(clip_index_path, clip_meta_path)
else:
    clip_index = RetrievalIndex.load(clip_index_path, clip_meta_path)

print('Both indices ready')

In [ ]:
# Compute Precision@K for both models on 20 test images
from src.models.colpali import ColPaliModel
from src.models.clip_model import CLIPEncoder

colpali_model = ColPaliModel()
colpali_model.load()

clip_enc = pipeline._clip  # reuse already loaded CLIP

n_eval = 20
colpali_retrieved_list = []
clip_retrieved_list    = []
relevant_paths_list    = []

for i in range(n_eval):
    img = test_images[i]
    gt_path = test_paths[i]

    # ColPali retrieval
    q_emb_cp = colpali_model.embed_query_image(img)
    colpali_retrieved_list.append(colpali_index.query(q_emb_cp, top_k=5))

    # CLIP retrieval
    q_emb_cl = clip_enc.embed_single_image(img)
    clip_retrieved_list.append(clip_index.query(q_emb_cl, top_k=5))

    relevant_paths_list.append([gt_path])

colpali_pak = compute_precision_at_k(colpali_retrieved_list, relevant_paths_list, k=5)
clip_pak    = compute_precision_at_k(clip_retrieved_list,    relevant_paths_list, k=5)

print(f'ColPali Precision@5: {colpali_pak:.4f}')
print(f'CLIP    Precision@5: {clip_pak:.4f}')

## 4. QA: with retrieval context vs without

In [ ]:
from src.mode2_qa import QAPipeline

qa_with_rag    = QAPipeline(index=colpali_index, retrieval_backend='colpali', top_k=5)
qa_with_rag.load_models()

# Without RAG: pass empty context
eval_qa_pairs = qa_pairs[:30]
preds_rag, preds_no_rag, gts = [], [], []

for pair in eval_qa_pairs:
    try:
        img = load_image(pair['image_path'])
        # With RAG
        r_rag = qa_with_rag.run(image=img, question=pair['question'])
        # Without RAG (empty context)
        ans_no_rag = qa_with_rag._medgemma.answer_question(image=img, question=pair['question'], context='')

        preds_rag.append(r_rag['answer'])
        preds_no_rag.append(ans_no_rag)
        gts.append(pair['answer'])
    except Exception as e:
        print(f'  Error: {e}')

metrics_rag    = compute_qa_metrics(preds_rag,    gts)
metrics_no_rag = compute_qa_metrics(preds_no_rag, gts)

print('\n=== QA Comparison ===')
print(f'With RAG (ColPali):  EM={metrics_rag["exact_match"]:.4f}, F1={metrics_rag["token_f1"]:.4f}')
print(f'Without RAG:         EM={metrics_no_rag["exact_match"]:.4f}, F1={metrics_no_rag["token_f1"]:.4f}')

## 5. Final comparison table

In [ ]:
all_model_results = {
    'MedGemma (Report Gen)': {
        'task': 'Report Generation',
        **medgemma_report_metrics,
    },
    'ColPali (Retrieval)': {
        'task': 'Retrieval',
        'precision_at_k': colpali_pak,
    },
    'CLIP (Retrieval)': {
        'task': 'Retrieval',
        'precision_at_k': clip_pak,
    },
    'MedGemma + ColPali RAG (QA)': {
        'task': 'QA with RAG',
        'exact_match': metrics_rag['exact_match'],
        'token_f1':    metrics_rag['token_f1'],
    },
    'MedGemma (QA, no RAG)': {
        'task': 'QA without RAG',
        'exact_match': metrics_no_rag['exact_match'],
        'token_f1':    metrics_no_rag['token_f1'],
    },
}

table_md = generate_comparison_table(all_model_results)
display(Markdown(table_md))

## 6. Qualitative side-by-side examples

In [ ]:
# Show 3 X-rays with: MedGemma report | GT report | Top ColPali retrieval score
n_examples = 3
fig, axes = plt.subplots(1, n_examples, figsize=(15, 5))

for i, ax in enumerate(axes):
    ax.imshow(test_images[i], cmap='gray')
    ax.set_title(f'X-ray {i+1}', fontsize=10)
    ax.axis('off')

plt.suptitle('Qualitative Examples — Test X-Rays', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

for i in range(n_examples):
    print(f"\n{'='*70}")
    print(f'Example {i+1}')
    print(f'\n[MedGemma Generated Report]\n{gen_reports[i][:500]}…')
    print(f'\n[Ground Truth Report]\n{test_reports[i][:500]}…')

## 7. Observations & Limitations

Fill this in after running the experiments — document:
- Where MedGemma performs well vs struggles
- ColPali vs CLIP retrieval: which finds better matches and why
- Effect of RAG on answer quality
- Failure cases

This section maps directly to the report's **Model Comparison** and **Conclusion** sections.